In [1]:
# test_march_retrieval.py
import os
import certifi
from dotenv import load_dotenv
from langchain_mongodb import MongoDBAtlasVectorSearch
from pymongo import MongoClient

load_dotenv()

EMBEDDING_PROVIDER = os.getenv("EMBEDDING_PROVIDER", "openai").lower()
if EMBEDDING_PROVIDER == "openai":
    from langchain_openai import OpenAIEmbeddings
    embedding = OpenAIEmbeddings(
        model="text-embedding-3-large",
        api_key=os.getenv("OPENAI_API_KEY"),
    )
elif EMBEDDING_PROVIDER == "gemini":
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    embedding = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=os.getenv("GEMINI_API_KEY"),
    )

client = MongoClient(os.getenv("MONGO_URI_ADMIN"), tlsCAFile=certifi.where())
collection = client["portfolio_rag"]["context_vectors"]

store = MongoDBAtlasVectorSearch(
    collection=collection,
    embedding=embedding,
    index_name="vector_index_context",
    text_key="text",
    embedding_key="embedding",
)

results = store.similarity_search("March 2026 market context portfolio", k=5)

print(f"Top 5 results for March 2026 query:\n")
for i, doc in enumerate(results):
    source = doc.metadata.get("original_filename") or doc.metadata.get("source", "unknown")
    print(f"Result {i+1}: {source}")
    print(f"  {doc.page_content[:150]}")
    print()

Top 5 results for March 2026 query:



In [ ]:
# check metadata
import os
import certifi
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv("MONGO_URI_ADMIN"), tlsCAFile=certifi.where())
col = client["portfolio_rag"]["context_vectors"]

total = col.count_documents({})
print(f"Total documents: {total}\n")

# ── Check source field patterns ────────────────────────────
print("=== Source Field Patterns ===")

# Documents with no source field at all
no_source = col.count_documents({"source": {"$exists": False}})
print(f"No source field:        {no_source}")

# Documents with /tmp/ paths (bad)
tmp_paths = col.count_documents({"source": {"$regex": "^/tmp/"}})
print(f"Temp /tmp/ paths:       {tmp_paths}")

# Documents with full local paths (from original FAISS migration)
local_paths = col.count_documents({"source": {"$regex": "^/Users/"}})
print(f"Local /Users/ paths:    {local_paths}")

# Documents with clean filenames (good)
clean = col.count_documents({
    "source": {
        "$exists": True,
        "$not": {"$regex": "^/tmp/"},
        "$not": {"$regex": "^/Users/"},
        "$not": {"$regex": "^/mount/"},
    }
})
print(f"Clean filenames:        {clean}")

# ── Check metadata field ───────────────────────────────────
print("\n=== Metadata Field ===")
no_metadata = col.count_documents({"metadata": None})
has_metadata = col.count_documents({"metadata": {"$ne": None}})
print(f"metadata is None:       {no_metadata}")
print(f"metadata has content:   {has_metadata}")

# ── Sample one document from each pattern ─────────────────
print("\n=== Sample Documents ===")

sample_local = col.find_one({"source": {"$regex": "^/Users/"}})
if sample_local:
    print(f"\nLocal path doc:")
    print(f"  source:   {sample_local.get('source')}")
    print(f"  metadata: {sample_local.get('metadata')}")

sample_clean = col.find_one({
    "source": {
        "$exists": True,
        "$not": {"$regex": "^/tmp/"},
        "$not": {"$regex": "^/Users/"},
    }
})
if sample_clean:
    print(f"\nClean doc:")
    print(f"  source:   {sample_clean.get('source')}")
    print(f"  metadata: {sample_clean.get('metadata')}")

In [ ]:
# check bad uploads
import os
import certifi
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv("MONGO_URI_ADMIN"), tlsCAFile=certifi.where())
db = client["portfolio_rag"]

collections = ["context_vectors", "pnl_vectors", "newsletter_vectors", "weekly_vectors"]

total_bad = 0
for name in collections:
    col = db[name]
    bad = col.count_documents({"source": {"$regex": "^/tmp/"}})
    total = col.count_documents({})
    total_bad += bad
    print(f"{name}: {bad} bad / {total} total")

print(f"\nTotal bad documents: {total_bad}")


In [ ]:
# check tmp files + remove
# Run once to remove all documents with temp file paths

import os, certifi
from pymongo import MongoClient
from dotenv import load_dotenv
load_dotenv()

client = MongoClient(os.getenv('MONGO_URI_ADMIN'), tlsCAFile=certifi.where())
col = client['portfolio_rag']['context_vectors']

# Preview files 
count = col.count_documents({'source': {'$regex': '^/tmp/'}})
print(f'Documents with temp paths: {count}')

# Uncomment to delete
# result = col.delete_many({'source': {'$regex': '^/tmp/'}})
# print(f'Deleted: {result.deleted_count}')

In [ ]:
import os
import certifi
from pathlib import Path
from datetime import datetime, timezone
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv("MONGO_URI_ADMIN"), tlsCAFile=certifi.where())
col = client["portfolio_rag"]["context_vectors"]

# ── Preview amount needs fixing ─────────────────────────────────
total = col.count_documents({})
needs_fix = col.count_documents({"metadata": None})
has_metadata = col.count_documents({"metadata": {"$ne": None}})

# Break down by source type
users_paths = col.count_documents({"source": {"$regex": "^/Users/"}})
tmp_paths = col.count_documents({"source": {"$regex": "^/tmp/"}})
no_source = col.count_documents({"source": {"$exists": False}})
other = total - users_paths - tmp_paths - no_source

# Break down what actually needs fixing
users_needs_fix = col.count_documents({
    "metadata": None,
    "source": {"$regex": "^/Users/"}
})
tmp_needs_fix = col.count_documents({
    "metadata": None,
    "source": {"$regex": "^/tmp/"}
})

print(f"{'='*40}")
print(f"TOTAL DOCUMENTS:        {total}")
print(f"{'='*40}")
print(f"\n--- Metadata Status ---")
print(f"metadata is None:       {needs_fix}")
print(f"metadata has content:   {has_metadata}")
print(f"\n--- Source Path Breakdown ---")
print(f"/Users/ paths:          {users_paths}  (original FAISS migration)")
print(f"/tmp/   paths:          {tmp_paths}  (bad admin uploads)")
print(f"no source field:        {no_source}")
print(f"other:                  {other}")
print(f"\n--- What This Script Will Fix ---")
print(f"Will update:            {users_needs_fix}  (/Users/ + metadata None)")
print(f"Will skip:              {tmp_needs_fix}  (/tmp/ — delete these separately)")
print(f"{'='*40}")
print()

In [2]:
# # ── Process in batches ─────────────────────────────────────

cursor = col.find(
    {
        "metadata": None,
        "source": {"$regex": "^/Users/"}  # ← only fix original migration docs
    },
    {"_id": 1, "source": 1, "text": 1},
    batch_size=500,
)


updated = 0
skipped = 0

for doc in cursor:
    source = doc.get("source", "")

    if not source:
        skipped += 1
        continue

    path = Path(source)
    filename = path.name  # e.g. "July 14 Fiscal and Tariff Uncertainties.pdf"

    # ── Infer collection category from source path ─────────
    # Matches your FOLDER_MAP in build_index.py
    source_lower = source.lower()
    if "context" in source_lower:
        collection_category = "context"
    elif "pnl" in source_lower:
        collection_category = "pnl"
    elif "newsletter" in source_lower:
        collection_category = "newsletter"
    elif "weekly" in source_lower:
        collection_category = "weekly_market_data"
    else:
        collection_category = "context"  # default

    # ── Infer file type ────────────────────────────────────
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        file_type = "pdf"
    elif suffix == ".csv":
        file_type = "csv"
    elif suffix == ".md":
        file_type = "markdown"
    elif suffix == ".txt":
        file_type = "text"
    else:
        file_type = "unknown"

    # ── Build metadata matching your upload tagging ────────
    metadata = {
        # Core fields — matches your admin_app.py upload tagging
        "source":             source,
        "original_filename":  filename,
        "uploaded_by":        "migration",   # marks these as pre-migration docs
        "collection":         collection_category,

        # Useful for filtering by time period in future queries
        # Extracted from filename where possible e.g. "July 14" → month hint
        "filename_date_hint": filename,  # agents can parse this if needed
    }

    col.update_one(
        {"_id": doc["_id"]},
        {"$set": {"metadata": metadata}}
    )

    updated += 1
    if updated % 500 == 0:
        print(f"  Updated {updated} / {needs_fix} documents...")

print(f"\nDone.")
print(f"  Updated: {updated}")
print(f"  Skipped: {skipped} (no source field)")

Total documents:        18437
Need metadata fix:      18437



In [ ]:
# ============================================================
# Test MongoDB connection 
# ============================================================
from pymongo import MongoClient
from dotenv import load_dotenv
import os
load_dotenv()
client = MongoClient(os.getenv('MONGO_URI_ADMIN'))
db = client[os.getenv('MONGO_DB_NAME', 'portfolio_rag')]
for name in db.list_collection_names():
    count = db[name].count_documents({})
    print(f'{name}: {count} documents')


In [ ]:
# ============================================================
# Test Auth 
# ============================================================
from auth_helper import verify_login
result = verify_login('admin', 'eTnoH$2001')
print(result)

# Expected:
# {'username': 'admin1', 'role': 'admin'}

In [ ]:
# ============================================================
# Test Vector Search, confirm connection to mongodb 
# ============================================================
from dotenv import load_dotenv
load_dotenv()
from multiagent import build_agent_system
import asyncio

orchestrator = build_agent_system()
result = asyncio.run(orchestrator.run_parallel('what are the key macro risks'))
print(result['market']['analysis'][:300])

In [ ]:
# testing source browsing -> showing newly uploaded files 

# Run this locally after uploading a file
import os, certifi
from pymongo import MongoClient
from dotenv import load_dotenv
load_dotenv()

client = MongoClient(os.getenv('MONGO_URI_ADMIN'), tlsCAFile=certifi.where())
col = client['portfolio_rag']['context_vectors']

# Total document count
print('Total documents:', col.count_documents({}))

# Most recently added document
import pymongo
doc = col.find_one(sort=[('_id', pymongo.DESCENDING)])
if doc:
    print('Most recent source:', doc.get('metadata', {}).get('original_filename', 'not found'))
    print('Also check source field:', doc.get('source', 'not found'))